In [ ]:
from pace_pic_validation.io.config import load_paths_config
from pace_pic_validation.io.csv_crunching import units_to_PIC_mmol_m3
import pandas as pd
import numpy as np
from pace_pic_validation.stats.regression import RobustRegression


MarSens = load_paths_config(config_name="MarSens_PIC.yaml")
SS_not_consolidated = load_paths_config(config_name = "SScruises_not_consolidated.yaml")
GNATS = load_paths_config(config_name="GNATS_PIC.yaml")
CHALKY = load_paths_config(config_name="Chalky.yaml")
output = load_paths_config(config_name= 'output_pic_path.yaml')


print(SS_not_consolidated['Cruises'])

print(CHALKY["inputs"]["pic_data"]["path"])



In [ ]:
#merging single cruise files (not consolidated)
cruises_separated = []



for cruise_name, cruise in SS_not_consolidated["Cruises"].items():
    
    ds_pic = pd.read_csv(cruise["pic_data"]["path"], skiprows = 6)
    ds_stations = pd.read_csv(cruise["station_reference"]["path"], skiprows = 23)
    print(ds_stations.columns)
    ds_stations['Time'] = ds_stations['Time'].str.replace('Z', '', regex=False)
    ds_stations['Time'] = pd.to_datetime(ds_stations['Time'], format='%Y-%m-%dT%H:%M:%S')
    ds_stations['Campaign'] = cruise_name
    ds = pd.merge(ds_pic, ds_stations, on="Station", how="left")
    cruises_separated.append(ds)


ds_SScruises = pd.concat(cruises_separated, ignore_index=True)

print(ds_SScruises.columns)

ds_SScruises = units_to_PIC_mmol_m3(ds_SScruises, start_unit="mg/m3", start_column="PIC_mg_per_m3", end_column="PIC_mmolC_m3", caco3=False)


ds_SScruises = ds_SScruises.drop(columns =['WaterTemperature_degC', 'Salinity_PSU', 'Conductivity_S_per_m', 'Depth_m', 'Chlorophylla_mg_per_m3',
       'TrueWindDir_deg', 'TrueWindSpeed_m_per_s', 'ShipAzimuth_deg',
       'AirTemperature_degC', 'AirPressure_bar', 'AirHumidity_percent',
       'BathymetryAboveMSL_m', 'WaterLevelAbsoveMSL_m', 'WaterColumn_m',
       'BottomWaterSpeed_m_per_s', 'SurfaceWaterSpeed_m_per_s',
       'Observations_y', 'Replicates_n', 'Observations_x', "PIC_mg_per_m3"])

print(ds_SScruises.columns)

ds_SScruises = ds_SScruises.rename(columns = {'Time': 'datetime', 
                                      'Latitude': 'lat',
                                      'Longitude': 'lon',
                                      } )
                                   
ds_SScruises['research_group'] = 'MarSens'

print(ds_SScruises.columns)



#accessing files
ds_pic = pd.read_csv(MarSens["inputs"]["pic_data"]["path"], skiprows=6)
ds_stations = pd.read_csv(MarSens["inputs"]["station_reference"]["path"], skiprows=7)

ds_MarSens = pd.merge(ds_pic, ds_stations, on="Station", how="left")

print(len(ds_MarSens))
print(ds_MarSens.iloc[0, 2])

ds_MarSens = ds_MarSens.loc[
    ~ds_MarSens["Station"].astype(str).str.strip().str.startswith("SS20240529", na=False)
].copy()

print(len(ds_MarSens))

# rename time_mean to datetime, Longitude_mean to longitude, Latitude_mean to latitude
ds_MarSens = ds_MarSens.rename(columns={"Time_mean": "datetime"})
ds_MarSens = ds_MarSens.rename(columns={"Longitude_mean": "lon"})
ds_MarSens = ds_MarSens.rename(columns={"Latitude_mean": "lat"})


# format datetime column to datetime object 

#remove Z at end of datetime string
ds_MarSens['datetime'] = ds_MarSens['datetime'].str.replace('Z', '', regex=False)
ds_MarSens['datetime'] = pd.to_datetime(ds_MarSens['datetime'], format='%Y-%m-%dT%H:%M:%S')

# add a column called research_group and fill it with the value "MarSens"

ds_MarSens['research_group'] = "MarSens"

print(MarSens['inputs']['pic_data']['unit'])

# format to mmol C /m3 
ds_MarSens = units_to_PIC_mmol_m3(ds_MarSens, start_unit="mg/m3", start_column="PIC_mg_per_m3", end_column="PIC_mmolC_m3", caco3=False)


ds_MarSens = ds_MarSens.drop(columns=["PIC_mg_per_m3","Replicates_n","Longitude_start", "Longitude_end", "Latitude_start", "Latitude_end", "Time_end", "Time_start"])



In [ ]:
# GNATS data formatting to same output as MarSens data
# data is in mol / m3 

ds_GNATS = pd.read_csv(GNATS["inputs"]["pic_data"]["path"])

# merging 'date' and 'time' columns into a single 'datetime' column and datetime[ns] format

time_str = ds_GNATS['time'].astype(str)
time_str = time_str.where(time_str.str.count(':') == 2, time_str + ':00')

ds_GNATS['datetime'] = pd.to_datetime(
    ds_GNATS['date'].astype(str) + ' ' + time_str,
    format='%Y%m%d %H:%M:%S',
    errors='coerce'
)

# add a research_group column and fill it with the value "GNATS"
ds_GNATS['research_group'] = "GNATS_Bigelow"

print(ds_GNATS['value'])

ds_GNATS['PIC_mmolC_m3'] = ds_GNATS['value'] * 1000




#drop unecessary columns
ds_GNATS = ds_GNATS.drop(columns=['value', "origin_file", "variable", "date", "time",'unit'])




#rename columns to match MarSens data
ds_GNATS = ds_GNATS.rename(columns={"station": "Station"})
ds_GNATS = ds_GNATS.rename(columns={"cruise": "Campaign"})

cols = ['Station', 'Campaign', 'datetime', 'lon', 'lat', 'research_group', 'PIC_mmolC_m3']

ds_GNATS = ds_GNATS[cols]
ds_MarSens = ds_MarSens[cols]
ds_SScruises = ds_SScruises[cols]

print(ds_GNATS.columns)
print(ds_MarSens.columns)
print(ds_SScruises.columns)


In [ ]:
# chalky dataset
# merge first because CTD and UW store position/time in different places

ds_pic_chalky = pd.read_csv(CHALKY["inputs"]["pic_data"]["path"])
ds_stations_chalky = pd.read_csv(CHALKY["inputs"]["station_reference"]["path"], skiprows=1)

print(ds_stations_chalky)

# build station key, fixing 0 in UW ID's
prefix = ds_pic_chalky["CTD / UW / Expt"].astype(str).str.strip()
ids = ds_pic_chalky["CTD ID / UW ID / Expt ID"].astype(str).str.strip()

ds_pic_chalky["Station"] = np.where(
    prefix.str.startswith("U"),
    prefix + ids.str.replace(r"^0+", "", regex=True),
    prefix + ids
)
# harmonize station column name in reference file
ds_stations_chalky = ds_stations_chalky.rename(columns={"STN\n": "Station"})


# merge
ds_merged = pd.merge(ds_pic_chalky, ds_stations_chalky, on="Station", how="left")


depth = ds_merged["Nominal Depth (m)"].astype(str).str.strip()

mask = (
    depth.str.contains("Surf", case=False, na=False) |
    (pd.to_numeric(depth, errors="coerce") < 7)
)

ds_merged = ds_merged[mask]
# creating one depth column to filter for surface data



print(ds_merged[["Station", "LAT N", "LON E", "Date +Time (GMT)"]].head(20))
print(ds_merged[["LAT N", "LON E", "Date +Time (GMT)"]].astype(str).head(20))
print(ds_merged[["LAT N", "LON E", "Date +Time (GMT)"]].isna().sum())


ds_merged.to_csv('/Volumes/Extreme SSD/PhD backup/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/PIC_discrete/ds_merged.csv')
# split:
# - UW rows: no Latitude from station table
# - CTD rows: Latitude present from station table


ds_UW = ds_merged[ds_merged["Latitude"].isna()].copy()
ds_UW = ds_UW.loc[ds_UW["LON E"].notna()].copy()
ds_Chalky = ds_merged[ds_merged["Latitude"].notna()].copy()

ds_UW.to_csv('/Volumes/Extreme SSD/PhD backup/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/PIC_discrete/ds_UW.csv')
ds_Chalky.to_csv('/Volumes/Extreme SSD/PhD backup/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/PIC_discrete/ds_Chalky.csv')

print("UW rows:", len(ds_UW))
print("CTD rows:", len(ds_Chalky))

# add metadata
for df in [ds_UW, ds_Chalky]:
    df["research_group"] = "Chalky"
    df["Campaign"] = "RV Discovery / Chalky"

# -------------------------
# UW branch
# coords/time come from UW-specific columns
# -------------------------
ds_UW = ds_UW.rename(columns={
    "LAT N": "lat",
    "LON E": "lon",
    "Date +Time (GMT)": "datetime",
    "PIC converted to [mmol C m-3]": "PIC_mmolC_m3"
})

# parse datetime if present
if "datetime" in ds_UW.columns:
    ds_UW["datetime"] = pd.to_datetime(ds_UW["datetime"], errors="coerce")

# drop unneeded columns for UW
uw_drop_cols = [
    "Date", "Latitude", "Longitude", "Event #", "CTD / UW / Expt",
    "CTD ID / UW ID / Expt ID", "Carboy #", "Niskin Bottle number",
    "Nominal Depth (m)", "DepSM", "PIC vol. (ml)",
    "Ghent Samples (500mL each)", "PIC Sample Tube Ref#", "Ca", "Sr", "Mg",
    "Na", "PIC ug L-1 (U-Maine analysis, Farley Calculation)",
    "Notes", "Unnamed: 20",
    "DATE", "Event", "TIME (GMT)", "Depth\n(m)", "Syn", "Peuk", "Nano",
    "Cocco", "Cryp", "Cpel", "HNA\nbac", "LNA\nbac", "Total bacteria",
    "Chl", "std Chl", "NO3+NO2", "SiO2", "NO2", "PO4", "SST",
    "PIC Maine", "Pic Ghent", "notes"
]
ds_UW = ds_UW.drop(columns=[c for c in uw_drop_cols if c in ds_UW.columns])

print("UW missing lat:", ds_UW["lat"].isna().sum() if "lat" in ds_UW.columns else "lat missing")
print("UW missing lon:", ds_UW["lon"].isna().sum() if "lon" in ds_UW.columns else "lon missing")
print("UW missing datetime:", ds_UW["datetime"].isna().sum() if "datetime" in ds_UW.columns else "datetime missing")

# -------------------------
# CTD branch
# coords come from merged station-reference columns
# -------------------------
ds_Chalky = ds_Chalky.rename(columns={
    "Latitude": "lat",
    "Longitude": "lon",
    "Date +Time (GMT)": "datetime",
    "PIC converted to [mmol C m-3]": "PIC_mmolC_m3"
})

if "datetime" in ds_Chalky.columns:
    ds_Chalky["datetime"] = pd.to_datetime(ds_Chalky["datetime"], errors="coerce")

ctd_drop_cols = [
    "Notes", "Unnamed: 20", "DATE", "Event", "TIME (GMT)",
    "LAT N", "LON E", "Depth\n(m)", "Syn", "Peuk", "Nano", "Cocco",
    "Cryp", "Cpel", "HNA\nbac", "LNA\nbac", "Total bacteria", "Chl",
    "std Chl", "NO3+NO2", "SiO2", "NO2", "PO4", "SST", "PIC Maine",
    "Pic Ghent", "notes", "Event #", "CTD / UW / Expt",
    "CTD ID / UW ID / Expt ID", "Carboy #", "Niskin Bottle number",
    "Nominal Depth (m)", "DepSM", "PIC vol. (ml)",
    "Ghent Samples (500mL each)", "PIC Sample Tube Ref#", "Ca", "Sr",
    "Mg", "Na", "PIC ug L-1 (U-Maine analysis, Farley Calculation)", "Date"
]
ds_Chalky = ds_Chalky.drop(columns=[c for c in ctd_drop_cols if c in ds_Chalky.columns])

print("CTD missing lat:", ds_Chalky["lat"].isna().sum() if "lat" in ds_Chalky.columns else "lat missing")
print("CTD missing lon:", ds_Chalky["lon"].isna().sum() if "lon" in ds_Chalky.columns else "lon missing")
print("CTD missing datetime:", ds_Chalky["datetime"].isna().sum() if "datetime" in ds_Chalky.columns else "datetime missing")

# -------------------------
# final shared schema
# make sure these names exist in both dfs
# -------------------------
cols = [
    "Station",
    "datetime",
    "lat",
    "lon",
    "PIC_mmolC_m3",
    "research_group",
    "Campaign"
]

# keep only columns that exist, but enforce same order
ds_UW = ds_UW[[c for c in cols if c in ds_UW.columns]].copy()
ds_Chalky = ds_Chalky[[c for c in cols if c in ds_Chalky.columns]].copy()

print("UW columns:", ds_UW.columns.tolist())
print("CTD columns:", ds_Chalky.columns.tolist())

In [ ]:
all_PIC = [ds_Chalky, ds_MarSens, ds_GNATS, ds_SScruises, ds_UW]

all_PIC = pd.concat(all_PIC, ignore_index=True)



all_PIC.to_csv(output['path'])

print(f'there are in total {len(all_PIC)} PIC observations and it is saved at{output['path']}')

